# Where are Rebel's officers now?

Check out the database

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("search_data_by_company_number.db")

pd.read_sql_query("SELECT * FROM search_data_by_company_number", conn)

,company_number,company_name,company_status,officer_id,office_name,officer_role,officer_appointed_on,officer_status,officer_resigned_on
0,10767623,REBEL ENERGY SUPPLY LIMITED,administration,7h2Kw6ljUYFhPbCb846YADzk4ag,Daniel Alan BATES,director,2019-11-12,active,NaN
1,10767623,REBEL ENERGY SUPPLY LIMITED,administration,W4ltg9kLyApR8j7K1NX8BYXb_Ic,Sharon Victoria DAWSON,director,2023-12-01,active,NaN
2,10767623,REBEL ENERGY SUPPLY LIMITED,administration,XOneXo1n1VUyuP-l5RmynDQJiXI,Bwalya KASASE,director,2022-05-12,active,NaN
3,10767623,REBEL ENERGY SUPPLY LIMITED,administration,n4RASDid7NTiER9ODSvEX95i3rI,Mark NEVEU,director,2023-11-21,active,NaN
4,10767623,REBEL ENERGY SUPPLY LIMITED,administration,OBbnHo3KqHNh0ebKrm1bT345f4o,Robert James Alexander TRIGGS,secretary,2022-05-26,resigned,2025-11-18
5,10767623,REBEL ENERGY SUPPLY LIMITED,administration,sanJ4ENm3vupRw5-MV5yDC_mnoM,"GOSLING, Steven Paul",director,2017-05-12,resigned,2019-11-12
6,10767623,REBEL ENERGY SUPPLY LIMITED,administration,uc0bIs52pzDfUumhh61vmZj8xEk,"GREEN, Andrew Michael",director,2017-05-12,resigned,2019-11-12
7,10767623,REBEL ENERGY SUPPLY LIMITED,administration,FT3ewwi4JLJD6LmizjPTmjqP2Co,"HIRST, Matthew Christopher",director,2017-05-12,resigned,2019-11-12
8,10767623,REBEL ENERGY SUPPLY LIMITED,administration,ONbG17h3BFk5lUznCBaXxT5Wp7A,"HOPE, Penelope Paterson",director,2020-07-27,resigned,2022-05-12
9,14583328,REBEL ENERGY SERVICES LTD,dissolved,7h2Kw6ljUYFhPbCb846YADzk4ag,Daniel Alan BATES,director,2023-01-10,inactive,NaN


In [2]:
from search_officers_by_company_number import (
    DATABASE_PATH,
    search_officers_by_company_number,
    store_data_from_search_by_company_number,
)
from search_companies_by_officer_id import (
    search_companies_by_officer_id,
    store_data_from_search_by_officer_id,
)


def find_active_company_rows(company_number, date):
    """Return active-company rows linked through recent/current officers."""
    company_number = str(company_number)
    cutoff_date = pd.to_datetime(date, errors="raise").date().isoformat()
    company_payload = search_officers_by_company_number(company_number)
    officer_payloads = []

    with sqlite3.connect(DATABASE_PATH) as conn:
        store_data_from_search_by_company_number(company_payload, conn)
        officer_ids = [
            row[0]
            for row in conn.execute(
                """
                SELECT DISTINCT officer_id
                FROM search_data_by_company_number
                WHERE company_number = ?
                  AND (
                      officer_resigned_on IS NULL
                      OR officer_resigned_on >= ?
                  )
                """,
                (company_number, cutoff_date),
            )
        ]

        for officer_id in officer_ids:
            officer_payload = search_companies_by_officer_id(officer_id)
            officer_payloads.append(officer_payload)
            store_data_from_search_by_officer_id(officer_payload, conn)

        payload_keys = sorted(
            {
                (company["company_number"], payload["officer_id"])
                for payload in officer_payloads
                for company in payload["companies"]
            }
        )
        if not payload_keys:
            return pd.read_sql_query(
                "SELECT * FROM search_data_by_company_number WHERE 0",
                conn,
            )

        values_sql = ", ".join("(?, ?)" for _ in payload_keys)
        params = [value for key in payload_keys for value in key]
        return pd.read_sql_query(
            f"""
            WITH payload_keys(company_number, officer_id) AS (
                VALUES {values_sql}
            )
            SELECT database_rows.*
            FROM search_data_by_company_number AS database_rows
            INNER JOIN payload_keys USING (company_number, officer_id)
            WHERE 1=1 
              AND database_rows.officer_status = 'active'
              AND database_rows.company_status = 'active'
            ORDER BY
                database_rows.company_number,
                database_rows.office_name
            """,
            conn,
            params=params,
        )


Are there any Rebel Energy officers now involved in other companies?

In [3]:
find_active_company_rows("10952085", "2025-06-01")

,company_number,company_name,company_status,officer_id,office_name,officer_role,officer_appointed_on,officer_status,officer_resigned_on
0,04870759,FOCUS COUNSELLING (UK) LTD,active,3jvSqTVwP8oSG2UoH1KGolr3BNM,Simon John Newton HEALE,director,2020-04-02,active,None


In [ ]:
businesses = pd.read_csv("non_compliant_businesses.csv")
businesses['breach_numbers_set'] = businesses['breach_numbers'].apply(eval).apply(set)

,business_name_sector,registered_address,breach_description,penalty_amount,appeal_status,breach_numbers,penalty_amount_pounds
0,Huckletree West Limited TCSP,"C/O Quantuma Advisory Limited, 7th Floor, 20 S...",Breach is for failure to apply for registratio...,"£5,720.00",No appeal,"(56,)",5720.0
1,Miles And Bird Limited EAB,"12 Bridge Road Bridge Road, East Molesey, KT8 9HA",Breach is for failure to apply for registratio...,"£5,500.00",No appeal,"(56,)",5500.0
2,Rent Proof Limited EAB,"6 Well Street, Birmingham, B19 3BG",Breach is for failure to apply for registratio...,"£3,200.00",No appeal,"(56,)",3200.0
3,Churchill Knight Umbrella Limited Accountancy ...,Suite G Hollies House 230 High Street Potters ...,Breach is for failure to apply for registratio...,"£52,000.00",No appeal,"(56,)",52000.0
4,R.A.S. Accounting Limited ASP,"4 Fore Street, Teignmouth, Devon, TQ14 8DZ",Breach is for failure to apply for registratio...,"£1,650.00",No appeal,"(56,)",1650.0
...,...,...,...,...,...,...,...
323,Javan Exchange Limited MSB,"141 Ballards Lane Finchley Central, London N3 1LJ",Breach is for failures in having the correct p...,"£15,483.00",No appeal,"(19, 33)",15483.0
324,Sharp Hawk UK Limited HVD,8 Legrace Avenue Hounselow Middlesex TW4 7RS,Breach is for failure to apply for registratio...,"£10,000.00",No appeal,"(56,)",10000.0
325,HB Clark & Co (Successors) Limited HVD,"Unit 3 Narvik Way, Tyne Tunnel Trading Estate,...",Breach is for failures in having the correct p...,"£28,126.00",No appeal,"(19, 28, 56)",28126.0
326,Gilmoora House Limited TCSP,"London House, 9a Margaret Street, London, W1W 8RJ",Breach is for failures in carrying out risk as...,"£18,436.00",No appeal,"(18, 19, 21, 24, 28, 30)",18436.0
